# Semana 4 · Cálculo y probabilidad para *deep learning*

**Nivel Intermedio** · Semana 4 · Clase expositiva [A] · 2 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem04_calculo.ipynb)

**Objetivos.** Al terminar la clase podrás:

1. Calcular derivadas, derivadas parciales y gradientes, a mano y numéricamente, y encadenarlas con la regla de la cadena.
2. Ver la pérdida (*loss*) como una función de los parámetros del modelo: los datos quedan fijos y lo que se mueve son los parámetros.
3. Programar el descenso de gradiente (*gradient descent*) a mano y explicar qué hace la tasa de aprendizaje (*learning rate*).
4. Convertir puntajes en probabilidades con *softmax* y medir el error de un clasificador con la entropía cruzada (*cross-entropy*).

**Prerrequisitos.** Semanas 2 y 3: arreglos de NumPy (formas, `axis`, *broadcasting*, `@`) y cargar un CSV con pandas. Saber qué es una derivada (es el requisito de entrada del curso; aquí la repasamos). Lectura previa: *Calculus basics for ML*, *Functions and Parameters* y *Probability and Distributions* de MÓLÓ. La clase parte de ahí y llega hasta el algoritmo con el que se entrena toda red neuronal.

**Agenda** (después de la apertura de 15 minutos).

| Parte | Tema | Minutos |
|---|---|---|
| 1 | La derivada: una pendiente | 15 |
| 2 | Derivadas parciales y gradiente | 15 |
| 3 | La regla de la cadena | 15 |
| 4 | La pérdida como función de los parámetros | 15 |
| 5 | Descenso de gradiente a mano | 20 |
| 6 | Probabilidad, *softmax* y entropía cruzada | 15 |
| 7 | Todo junto: un clasificador entrenado con gradiente | 5 |
| 8 | Cierre y tareas para la semana 5 | 5 |

Ejecuta esta celda primero: carga las librerías, fija la semilla y define `check`, la función que revisa tus ejercicios de "Tu turno".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


def central_diff(f, x, h=1e-5):
    """Derivada numérica de referencia, para las celdas de verificación."""
    return (f(x + h) - f(x - h)) / (2 * h)


print("NumPy", np.__version__)

## 1. La derivada: una pendiente

La derivada $f'(x)$ dice **cuánto cambia $f$ cuando $x$ se mueve un poquito**. Es la pendiente de la curva en ese punto. Piensa en el velocímetro de un auto: no dice dónde estás, sino qué tan rápido cambia tu posición en ese instante.

$$f'(x) \approx \frac{f(x + h) - f(x)}{h} \qquad \text{con } h \text{ pequeño}$$

Un ejemplo con números: $f(x) = x^2$ en $x = 3$. La regla de la potencia dice $f'(x) = 2x$, así que $f'(3) = 6$: cerca de $x = 3$, si $x$ sube $0.001$, $f$ sube unas seis veces más, $0.006$. Comprobémoslo achicando $h$:

In [ ]:
def f(x):
    return x**2


for h in [1.0, 0.1, 0.01, 0.001]:
    slope = (f(3 + h) - f(3)) / h
    print(f"h = {h:<6} → pendiente ≈ {slope:.4f}")

La aproximación mejora al achicar $h$, pero no sin límite. Con un $h$ demasiado chico, $f(3 + h)$ y $f(3)$ son casi iguales y la resta pierde justo los dígitos que importan: la computadora guarda unos 16 dígitos por número (*float64*).

La **diferencia centrada** mira a los dos lados de $x$ y, con el mismo $h$, es mucho más precisa:

$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$$

Comparemos el error de las dos fórmulas con $f(x) = e^x$ en $x = 1$, cuya derivada exacta es $e^1$:

In [ ]:
exact = np.exp(1.0)
print(f"{'h':>8}{'error hacia adelante':>24}{'error centrada':>18}")
for h in [1e-1, 1e-3, 1e-5, 1e-8, 1e-12]:
    forward = (np.exp(1 + h) - np.exp(1)) / h
    central = (np.exp(1 + h) - np.exp(1 - h)) / (2 * h)
    print(f"{h:>8.0e}{abs(forward - exact):>24.2e}{abs(central - exact):>18.2e}")

La centrada con $h \approx 10^{-5}$ acierta unos 10 dígitos; con $h = 10^{-12}$ las dos empeoran por el redondeo. Es la versión que usaremos todo el año para **verificar** derivadas calculadas a mano (en la semana 7, para verificar *backpropagation*).

Las reglas que vas a necesitar:

| Función | Derivada |
|---|---|
| $x^n$ | $n\,x^{n-1}$ |
| $e^x$ | $e^x$ |
| $\ln x$ | $1/x$ |
| $c \cdot f(x)$ | $c \cdot f'(x)$ |
| $f(x) + g(x)$ | $f'(x) + g'(x)$ |

**La recta tangente.** Cerca de un punto $a$, toda función suave se parece a una recta: $f(x) \approx f(a) + f'(a)\,(x - a)$. Esa idea, "de cerca, todo es una recta", es la que usa el descenso de gradiente. Y donde la función tiene un mínimo, la tangente es horizontal: **la derivada vale 0**.

In [ ]:
xs = np.linspace(-3, 4.5, 200)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, f(xs), lw=3, label="$f(x) = x^2$")
for a, color in [(-1, "tab:green"), (0, "tab:purple"), (3, "tab:orange")]:
    slope = 2 * a
    near = np.linspace(a - 1.5, a + 1.5, 10)
    ax.plot(near, f(a) + slope * (near - a), "--", color=color, lw=2, label=f"tangente en x = {a}: pendiente {slope}")
    ax.plot(a, f(a), "o", color=color, ms=8)
ax.set_ylim(-2, 20)
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.set_title("La derivada es la pendiente de la tangente")
ax.legend()
plt.show()

### Tu turno 1: la derivada numérica

Completa `numerical_derivative(f, x, h=1e-5)` con la diferencia centrada. Debe funcionar tanto si `x` es un número como si es un arreglo de NumPy (si usas solo operaciones aritméticas, sale solo).

La celda de verificación la compara con derivadas que conoces: $\sin' = \cos$, $(x^3)' = 3x^2$ y $(e^x)' = e^x$.

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    # TODO
    raise NotImplementedError

In [ ]:
def test_numerical_derivative():
    pts = np.array([0.0, 1.0, 2.5])
    assert np.allclose(numerical_derivative(np.sin, pts), np.cos(pts)), "la derivada de sin debe ser cos"
    assert np.isclose(numerical_derivative(lambda x: x**3, 2.0), 12.0), "la derivada de x³ en 2 debe ser 12"
    assert np.isclose(numerical_derivative(np.exp, 0.0), 1.0), "la derivada de eˣ en 0 debe ser 1"


check("numerical_derivative", test_numerical_derivative)

## 2. Derivadas parciales y gradiente

Un modelo tiene muchos parámetros, así que su pérdida es una función de **muchas variables**. La **derivada parcial** $\partial f / \partial x$ es la derivada respecto de una sola variable **dejando todas las demás quietas**.

Ejemplo: $f(x, y) = x^2 + 3y^2$.

- Con $y$ quieta, $3y^2$ es una constante: $\dfrac{\partial f}{\partial x} = 2x$.
- Con $x$ quieta: $\dfrac{\partial f}{\partial y} = 6y$.

En el punto $(1, 2)$ valen $2$ y $12$: moverse en $y$ cambia $f$ seis veces más que moverse en $x$.

El **gradiente** junta todas las derivadas parciales en un vector:

$$\nabla f(x, y) = \left( \frac{\partial f}{\partial x}, \frac{\partial f}{\partial y} \right) = (2x,\; 6y), \qquad \nabla f(1, 2) = (2, 12)$$

Imagina que estás en una montaña con niebla. No ves el valle, pero con los pies sientes hacia dónde se inclina el suelo. Eso es el gradiente: una información **local** que dice hacia dónde **sube** más rápido la función. Su opuesto, $-\nabla f$, dice hacia dónde baja más rápido.

In [ ]:
def f2(x, y):
    return x**2 + 3 * y**2


h = 1e-5
dfdx = (f2(1 + h, 2) - f2(1 - h, 2)) / (2 * h)  # solo se mueve x
dfdy = (f2(1, 2 + h) - f2(1, 2 - h)) / (2 * h)  # solo se mueve y
print(f"∂f/∂x en (1, 2) ≈ {dfdx:.4f}   (a mano: 2x = 2)")
print(f"∂f/∂y en (1, 2) ≈ {dfdy:.4f}   (a mano: 6y = 12)")

Un **mapa de curvas de nivel** (*contour plot*) dibuja las líneas donde $f$ vale lo mismo, como un mapa topográfico. Las flechas son el gradiente en varios puntos: siempre cruzan las curvas de nivel en ángulo recto, apuntando cuesta arriba.

In [ ]:
gx, gy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-2, 2, 200))
px, py = np.meshgrid(np.linspace(-2.5, 2.5, 7), np.linspace(-1.6, 1.6, 5))

fig, ax = plt.subplots(figsize=(7, 4.5))
cs = ax.contour(gx, gy, f2(gx, gy), levels=[0.5, 2, 4, 7, 11, 16], cmap="viridis")
ax.clabel(cs, fontsize=8)
ax.quiver(px, py, 2 * px, 6 * py, color="tab:red", angles="xy", width=0.004)
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Curvas de nivel de $x^2 + 3y^2$ y su gradiente (flechas)")
plt.show()

### Tu turno 2: el gradiente, a mano y numérico

Trabajamos con $g(x, y) = x^2 y + 3y$. Los puntos se guardan como arreglos `p = np.array([x, y])`.

1. `grad_g(p)`: el gradiente **calculado a mano**, como arreglo `np.array([∂g/∂x, ∂g/∂y])`.
2. `numerical_gradient(f, p, h=1e-5)`: el gradiente numérico de **cualquier** función `f` que recibe un arreglo `p` de cualquier largo. Para cada coordinada `i`, mueve solo `p[i]` hacia arriba y hacia abajo y usa la diferencia centrada. Aquí sí puedes usar un bucle sobre las coordenadas. Cuidado: trabaja sobre copias de `p` (`p.astype(float)` crea una), no lo modifiques.

In [ ]:
def g(p):
    x, y = p
    return x**2 * y + 3 * y


def grad_g(p):
    # TODO
    raise NotImplementedError


def numerical_gradient(f, p, h=1e-5):
    # TODO
    raise NotImplementedError

In [ ]:
def test_grad_g():
    assert np.allclose(grad_g(np.array([1.0, 2.0])), [4.0, 4.0]), "en (1, 2) el gradiente de g es (4, 4)"
    assert np.allclose(grad_g(np.array([-3.0, 0.5])), [-3.0, 12.0]), "revisa ∂g/∂x y ∂g/∂y"


def test_numerical_gradient():
    for _ in range(5):
        p = rng.normal(size=2)
        assert np.allclose(numerical_gradient(g, p), [2 * p[0] * p[1], p[0] ** 2 + 3], atol=1e-6), "no coincide con el gradiente de g"
    q = rng.normal(size=4)
    q_before = q.copy()
    assert np.allclose(numerical_gradient(lambda v: np.sum(v**2), q), 2 * q, atol=1e-6), "el gradiente de Σ v² es 2v"
    assert np.array_equal(q, q_before), "numerical_gradient no debe modificar p"


check("grad_g", test_grad_g)
check("numerical_gradient", test_numerical_gradient)

## 3. La regla de la cadena

Una red neuronal es una función dentro de otra, dentro de otra. Para derivar una composición $y = f(u)$ con $u = g(x)$ se multiplican las derivadas de cada eslabón:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

Piensa en engranajes: si cada vuelta del engranaje A hace girar al B **3** veces, y cada vuelta del B hace girar al C **2** veces, entonces cada vuelta del A hace girar al C $3 \times 2 = 6$ veces. Las tasas de cambio se multiplican.

Ejemplo: $h(x) = (3x + 1)^2$. Con $u = 3x + 1$ queda $h = u^2$:

$$\frac{dh}{dx} = \underbrace{2u}_{dh/du} \cdot \underbrace{3}_{du/dx} = 6\,(3x + 1), \qquad \text{en } x = 1: \; 6 \cdot 4 = 24$$

In [ ]:
def h_func(x):
    return (3 * x + 1) ** 2


print("regla de la cadena:", 6 * (3 * 1 + 1))
print("numérica:          ", round(central_diff(h_func, 1.0), 6))

### Una neurona, eslabón por eslabón

Una neurona con una entrada $x$, un peso $w$ y un sesgo $b$ calcula $z = wx + b$ y después aplica la **sigmoide** $\sigma(z) = \dfrac{1}{1 + e^{-z}}$, que aplasta cualquier número entre 0 y 1. Su error, comparado con la respuesta $y$, es $L = (a - y)^2$. Tres eslabones:

$$w \;\longrightarrow\; z = wx + b \;\longrightarrow\; a = \sigma(z) \;\longrightarrow\; L = (a - y)^2$$

La derivada de la sigmoide tiene una forma cómoda: $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$. Con la regla de la cadena:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w} = 2(a - y) \cdot \sigma(z)\,(1 - \sigma(z)) \cdot x$$

Con números pequeños: $x = 2$, $y = 1$, $w = 0.5$, $b = -1$. Entonces $z = 0$, $a = 0.5$ y $L = 0.25$. Los tres factores valen $-1$, $0.25$ y $2$, así que $\partial L / \partial w = -0.5$: si $w$ sube un poquito, el error **baja** la mitad de eso.

Calcular cada derivada local y multiplicarlas de atrás hacia adelante es exactamente lo que hace *backpropagation*, que veremos despacio en la semana 7.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


x0, y0, w0, b0 = 2.0, 1.0, 0.5, -1.0
z = w0 * x0 + b0
a = sigmoid(z)
dL_da = 2 * (a - y0)
da_dz = a * (1 - a)
dz_dw = x0
print(f"z = {z}, a = {a}, L = {(a - y0) ** 2}")
print(f"factores: {dL_da} · {da_dz} · {dz_dw} = {dL_da * da_dz * dz_dw}")

loss_of_w = lambda w: (sigmoid(w * x0 + b0) - y0) ** 2
print("numérica:", round(central_diff(loss_of_w, w0), 6))

### Tu turno 3: las dos derivadas de la neurona

Completa `neuron_grads(w, b, x, y)`: devuelve la tupla `(dL_dw, dL_db)` para la neurona de arriba, $L = (\sigma(wx + b) - y)^2$. Para $\partial L / \partial b$ cambia solo el último eslabón: ¿cuánto vale $\partial z / \partial b$?

Con los números de arriba debe dar `(-0.5, -0.25)`.

In [ ]:
def neuron_grads(w, b, x, y):
    # TODO
    raise NotImplementedError

In [ ]:
def test_neuron_grads():
    dw, db = neuron_grads(0.5, -1.0, 2.0, 1.0)
    assert np.isclose(dw, -0.5) and np.isclose(db, -0.25), f"con los números del ejemplo dio ({dw}, {db}), esperaba (-0.5, -0.25)"
    for _ in range(5):
        w, b, x, y = rng.normal(size=4)
        dw, db = neuron_grads(w, b, x, y)
        num_w = central_diff(lambda v: (sigmoid(v * x + b) - y) ** 2, w)
        num_b = central_diff(lambda v: (sigmoid(w * x + v) - y) ** 2, b)
        assert np.isclose(dw, num_w, atol=1e-7), "dL_dw no coincide con la derivada numérica"
        assert np.isclose(db, num_b, atol=1e-7), "dL_db no coincide con la derivada numérica"


check("neuron_grads", test_neuron_grads)

## 4. La pérdida como función de los parámetros

Volvemos a los pingüinos de la semana 3. Pregunta: ¿se puede predecir la masa de un pingüino a partir del largo de su aleta? El modelo más simple es una recta:

$$\hat{y} = w\,x + b$$

- $x$: el largo de la aleta **estandarizado** (le restamos la media y dividimos por la desviación estándar, como en el B1), para que quede alrededor de 0.
- $y$: la masa en kilogramos.

Para medir qué tan mala es una recta usamos el **error cuadrático medio** (*mean squared error*, MSE): el promedio de los errores al cuadrado.

$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} \left(w\,x_i + b - y_i\right)^2$$

Fíjate en qué es variable aquí: **los datos están fijos**, lo que cambia son los parámetros $w$ y $b$. Cada elección de $(w, b)$ es una recta distinta y tiene su propio error. Entrenar el modelo es buscar el $(w, b)$ donde $L$ es más chica.

**Línea base**: la recta horizontal $w = 0$, $b = \bar{y}$, que predice siempre la masa promedio sin mirar la aleta. Su MSE es la varianza de $y$.

In [ ]:
penguins = pd.read_csv(URL).dropna(subset=["flipper_length_mm", "body_mass_g", "bill_length_mm", "species"])
flipper = penguins["flipper_length_mm"].to_numpy()
x = (flipper - flipper.mean()) / flipper.std()  # aleta estandarizada
y = penguins["body_mass_g"].to_numpy() / 1000   # masa en kg
print("n =", len(x))


def mse(w, b):
    return np.mean((w * x + b - y) ** 2)


baseline_mse = mse(0.0, y.mean())
print(f"línea base (siempre {y.mean():.2f} kg): MSE = {baseline_mse:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=12, alpha=0.5, color="gray")
xs = np.linspace(x.min(), x.max(), 10)
for w, b, color in [(0.0, y.mean(), "tab:gray"), (0.3, 3.8, "tab:red"), (0.7, 4.2, "tab:green")]:
    ax.plot(xs, w * xs + b, color=color, lw=3, label=f"w = {w}, b = {b:.2f}: MSE = {mse(w, b):.3f}")
ax.set_xlabel("largo de la aleta (estandarizado)")
ax.set_ylabel("masa (kg)")
ax.set_title("Tres rectas, tres errores")
ax.legend()
plt.show()

Si calculamos $L$ para muchos valores de $w$ y $b$, obtenemos un **paisaje**: cada punto del mapa es una recta, y la altura es su error. El punto más bajo es la mejor recta. Para la recta existe una fórmula cerrada (la de mínimos cuadrados, que es lo que calcula `np.polyfit`), pero para una red neuronal no la hay. Por eso aprendemos a bajar por el paisaje.

In [ ]:
w_best, b_best = np.polyfit(x, y, 1)
print(f"mínimo (np.polyfit): w = {w_best:.3f}, b = {b_best:.3f}, MSE = {mse(w_best, b_best):.3f}")

W_grid, B_grid = np.meshgrid(np.linspace(-1, 2.4, 150), np.linspace(2.0, 6.0, 150))
L_grid = np.mean((W_grid[..., None] * x + B_grid[..., None] - y) ** 2, axis=-1)  # (150, 150, n) → (150, 150)

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contour(W_grid, B_grid, L_grid, levels=[0.2, 0.3, 0.5, 0.8, 1.2, 1.8, 2.6, 3.6], cmap="viridis")
ax.clabel(cs, fontsize=8)
ax.plot(w_best, b_best, "*", color="tab:orange", ms=18, label="mínimo")
ax.plot(0, y.mean(), "o", color="tab:gray", ms=9, label="línea base")
ax.set_xlabel("w (pendiente)")
ax.set_ylabel("b (sesgo)")
ax.set_title("El paisaje de la pérdida L(w, b)")
ax.legend()
plt.show()

### Tu turno 4: el gradiente de la pérdida

Completa `mse_grad(w, b, x, y)`: devuelve `(dL_dw, dL_db)`, las derivadas parciales del MSE, **sin bucles**.

Pista, con la regla de la cadena: llama $r_i = w x_i + b - y_i$ al error del ejemplo $i$. La derivada de $r_i^2$ es $2 r_i$ por la derivada de $r_i$, y $\partial r_i / \partial w = x_i$, $\partial r_i / \partial b = 1$. Después promedia sobre los ejemplos.

En el mínimo, las dos derivadas deben dar (casi) 0.

In [ ]:
def mse_grad(w, b, x, y):
    # TODO
    raise NotImplementedError

In [ ]:
def test_mse_grad():
    for w, b in [(0.0, 0.0), (1.5, 3.0), (-0.4, 5.2)]:
        dw, db = mse_grad(w, b, x, y)
        num_w = central_diff(lambda v: np.mean((v * x + b - y) ** 2), w)
        num_b = central_diff(lambda v: np.mean((w * x + v - y) ** 2), b)
        assert np.isclose(dw, num_w, atol=1e-6), f"en (w, b) = ({w}, {b}), dL_dw = {dw:.4f} y la numérica da {num_w:.4f}"
        assert np.isclose(db, num_b, atol=1e-6), f"en (w, b) = ({w}, {b}), dL_db = {db:.4f} y la numérica da {num_b:.4f}"
    assert np.allclose(mse_grad(w_best, b_best, x, y), 0, atol=1e-8), "en el mínimo el gradiente debe ser 0"


check("mse_grad", test_mse_grad)

## 5. Descenso de gradiente a mano

Volvamos a la montaña con niebla. Una estrategia para llegar al valle: sentir hacia dónde baja el suelo, dar un paso en esa dirección, y repetir. Eso es el **descenso de gradiente**. Con $\theta$ para todos los parámetros juntos:

$$\theta \;\leftarrow\; \theta - \eta \, \nabla L(\theta)$$

El número $\eta$ (eta) es el largo del paso: la **tasa de aprendizaje** (*learning rate*). No lo aprende el modelo, lo elegimos nosotros: es un **hiperparámetro** (*hyperparameter*).

Primero en una dimensión, con $L(w) = (w - 3)^2$, cuyo mínimo está en $w = 3$. Su derivada es $L'(w) = 2(w - 3)$. Desde $w = 0$ con $\eta = 0.1$:

- paso 1: $w = 0 - 0.1 \cdot 2(0 - 3) = 0.6$
- paso 2: $w = 0.6 - 0.1 \cdot 2(0.6 - 3) = 1.08$
- … y cada paso se acerca un 20% de lo que falta.

In [ ]:
def L1(w):
    return (w - 3) ** 2


def dL1(w):
    return 2 * (w - 3)


w = 0.0
lr = 0.1
for step in range(1, 31):
    w = w - lr * dL1(w)
    if step <= 4 or step % 10 == 0:
        print(f"paso {step:2d}: w = {w:.4f}   L = {L1(w):.6f}")

### Elegir la tasa de aprendizaje

¿Qué pasa con otros valores de $\eta$? Aquí van cuatro, cada uno con 15 pasos desde $w = 0$:

- muy chica: avanza, pero tan lento que no llega;
- buena: llega en pocos pasos;
- grande: se pasa al otro lado del valle y rebota, aunque termina llegando;
- demasiado grande: cada rebote es más alto que el anterior y **diverge**.

Para esta función se puede ver por qué: cada paso multiplica la distancia al mínimo, $w - 3$, por $(1 - 2\eta)$. Si $|1 - 2\eta| < 1$, es decir $0 < \eta < 1$, la distancia se achica; si no, crece. En el B2 vas a buscar ese límite en otras funciones.

In [ ]:
def descend_1d(dL, w0, lr, steps):
    path = [w0]
    for _ in range(steps):
        path.append(path[-1] - lr * dL(path[-1]))
    return np.array(path)


fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
ws = np.linspace(-1.5, 7.5, 200)
for ax, lr, title in zip(axes, [0.02, 0.3, 0.9, 1.05], ["muy chica", "buena", "grande: rebota", "demasiado grande: diverge"]):
    path = descend_1d(dL1, 0.0, lr, 15)
    ax.plot(ws, L1(ws), color="tab:blue", lw=2)
    ax.plot(path, L1(path), "o-", color="tab:orange", ms=4, lw=1)
    ax.set_title(f"η = {lr}: {title}")
    ax.set_xlim(-1.5, 7.5)
    ax.set_ylim(-1, 25)
    ax.set_xlabel("w")
axes[0].set_ylabel("L(w)")
plt.tight_layout()
plt.show()

### En dos dimensiones: la recta de los pingüinos

La regla es la misma con dos parámetros: en cada paso, $w$ y $b$ se mueven a la vez en contra de su derivada parcial. Empezamos lejos, en $w = -0.5$, $b = 2.5$, con $\eta = 0.1$ y 60 pasos. Esta celda funciona cuando completaste el Tu turno 4.

In [ ]:
try:
    w, b = -0.5, 2.5
    lr = 0.1
    path, losses = [(w, b)], [mse(w, b)]
    for step in range(60):
        dw, db = mse_grad(w, b, x, y)
        w, b = w - lr * dw, b - lr * db
        path.append((w, b))
        losses.append(mse(w, b))
    path = np.array(path)

    print(f"descenso de gradiente: w = {w:.3f}, b = {b:.3f}, MSE = {mse(w, b):.3f}")
    print(f"np.polyfit:            w = {w_best:.3f}, b = {b_best:.3f}, MSE = {mse(w_best, b_best):.3f}")
    print(f"línea base:                                MSE = {baseline_mse:.3f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
    cs = ax1.contour(W_grid, B_grid, L_grid, levels=[0.2, 0.3, 0.5, 0.8, 1.2, 1.8, 2.6, 3.6], cmap="viridis")
    ax1.plot(path[:, 0], path[:, 1], "o-", color="tab:orange", ms=3, lw=1, label="descenso de gradiente")
    ax1.plot(w_best, b_best, "*", color="tab:red", ms=16, label="mínimo")
    ax1.set_xlabel("w")
    ax1.set_ylabel("b")
    ax1.set_title("El camino sobre el paisaje")
    ax1.legend()
    ax2.plot(losses, color="tab:orange", lw=2, label="descenso de gradiente")
    ax2.axhline(baseline_mse, color="gray", ls="--", label="línea base")
    ax2.set_xlabel("paso")
    ax2.set_ylabel("MSE")
    ax2.set_title("La pérdida en cada paso (curva de aprendizaje)")
    ax2.legend()
    plt.tight_layout()
    plt.show()
except NotImplementedError:
    print("Completa mse_grad en el Tu turno 4 para ver el descenso de gradiente.")

El camino baja perpendicular a las curvas de nivel, como las flechas de la parte 2, y llega al mismo mínimo que `np.polyfit` sin usar ninguna fórmula especial: solo el gradiente. Ese gráfico de la derecha, la pérdida paso a paso, se llama **curva de aprendizaje** (*learning curve*): lo vas a mirar en cada entrenamiento del año.

## 6. Probabilidad, *softmax* y entropía cruzada

Para **clasificar** (¿qué especie es este pingüino?) el modelo no da un número, sino una **distribución de probabilidad** sobre las clases: números entre 0 y 1 que suman 1. La más simple es contar: la fracción de cada especie en los datos.

In [ ]:
freqs = penguins["species"].value_counts(normalize=True)
print(freqs.round(3))
print("suma:", freqs.sum())
print(f"línea base (siempre {freqs.index[0]}): accuracy {freqs.iloc[0]:.3f}")

### De puntajes a probabilidades: *softmax*

Una red termina con un **puntaje** por clase (*logits*), que puede ser cualquier número, negativo o mayor que 1. La ***softmax*** los convierte en probabilidades: primero cada puntaje pasa por $e^z$, que lo vuelve positivo, y después se divide por la suma, para que sumen 1.

$$\text{softmax}(z)_j = \frac{e^{z_j}}{\sum_k e^{z_k}}$$

Ejemplo: $z = (2, 1, 0.1)$. Entonces $e^z \approx (7.39, 2.72, 1.11)$, que suman $11.21$, y las probabilidades son $(0.66, 0.24, 0.10)$. Como una votación convertida en porcentajes, pero la exponencial exagera las diferencias: el que más puntos tiene se lleva una tajada desproporcionada.

Dos propiedades. Sumar la misma constante a todos los puntajes no cambia nada (se cancela arriba y abajo), y por eso restamos el máximo: evita que $e^{1000}$ se desborde. Multiplicar los puntajes por 10 hace al modelo mucho más seguro.

In [ ]:
def softmax(Z):
    """Softmax de cada fila de Z, con forma (n, k)."""
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


z = np.array([[2.0, 1.0, 0.1]])
print("softmax(z):       ", softmax(z).round(3))
print("softmax(z + 100): ", softmax(z + 100).round(3), " ← igual")
print("softmax(10 · z):  ", softmax(10 * z).round(3), " ← más seguro")
print("softmax([1000, 1001, 999]):", softmax(np.array([[1000.0, 1001.0, 999.0]])).round(3), " ← sin desbordarse")

### Por qué el logaritmo

Para juzgar un clasificador miramos la probabilidad que le dio a la **respuesta correcta** de cada ejemplo. La probabilidad de acertar todos a la vez es el producto de esas probabilidades, y un producto de cientos de números menores que 1 se vuelve tan chico que la computadora lo redondea a 0. El logaritmo resuelve eso: convierte productos en sumas, $\ln(ab) = \ln a + \ln b$.

Además, $-\ln p$ se lee como **sorpresa**: si el modelo le daba probabilidad 1 a la respuesta correcta, la sorpresa es 0; si le daba 0.01, la sorpresa es 4.6. Equivocarse con mucha confianza cuesta caro.

In [ ]:
probs = np.full(2000, 0.5)
print("producto de 2000 probabilidades de 0.5:", np.prod(probs), " ← se redondeó a 0")
print("suma de sus logaritmos:                ", np.sum(np.log(probs)))
print()
for p in [1.0, 0.9, 0.5, 0.1, 0.01]:
    print(f"p = {p:<5} → sorpresa −ln p = {-np.log(p) + 0:.3f}")

### La entropía cruzada

La **entropía cruzada** (*cross-entropy*) es la sorpresa promedio sobre todos los ejemplos. Si `P` tiene forma `(n, k)` (una fila de probabilidades por ejemplo) y $y_i$ es la clase correcta del ejemplo $i$:

$$L = -\frac{1}{n} \sum_{i=1}^{n} \ln P_{i,\,y_i}$$

Solo importa la probabilidad que el modelo le dio a la clase correcta. Es la pérdida de casi todos los clasificadores que vas a entrenar este año.

**Líneas base para la pérdida.** Un modelo que no sabe nada y reparte la probabilidad por igual entre $k$ clases tiene pérdida $\ln k$: con 3 clases, $\ln 3 \approx 1.099$. Si tu clasificador no baja de ahí, no aprendió nada.

### Tu turno 5: la entropía cruzada

Completa `cross_entropy(P, y)`, **sin bucles**. Pista: indexación con arreglos, como en el B1. `P[np.arange(n), y]` elige, en cada fila `i`, la columna `y[i]`.

In [ ]:
def cross_entropy(P, y):
    # TODO
    raise NotImplementedError

In [ ]:
def test_cross_entropy():
    P = np.array([[0.7, 0.2, 0.1], [0.1, 0.1, 0.8]])
    expected = -(np.log(0.7) + np.log(0.8)) / 2
    assert np.isclose(cross_entropy(P, np.array([0, 2])), expected), "revisa el ejemplo pequeño"
    uniform = np.full((50, 4), 0.25)
    assert np.isclose(cross_entropy(uniform, rng.integers(0, 4, 50)), np.log(4)), "con probabilidades uniformes debe dar ln k"
    assert np.isclose(cross_entropy(np.eye(3), np.array([0, 1, 2])), 0), "un modelo perfecto tiene pérdida 0"


check("cross_entropy", test_cross_entropy)

### El gradiente de la entropía cruzada

Juntando *softmax* y entropía cruzada, la derivada de la pérdida respecto de los puntajes $Z$ es sorprendentemente simple:

$$\frac{\partial L}{\partial Z} = \frac{1}{n}\,(P - Y)$$

donde $Y$ es la codificación *one-hot* de las etiquetas (un 1 en la clase correcta, 0 en las demás). Se lee así: a la clase correcta, el gradiente le sube el puntaje en lo que le faltó para llegar a 1; a las demás se lo baja en la probabilidad que se llevaron por error. La demostración es un buen ejercicio de regla de la cadena (la haremos en la semana 7); hoy la comprobamos numéricamente.

In [ ]:
Z = rng.normal(size=(4, 3))
labels = np.array([0, 2, 1, 2])
Y = np.eye(3)[labels]  # one-hot, (4, 3)


def ce_of_Z(Z):
    return -np.mean(np.log(softmax(Z)[np.arange(len(labels)), labels]))


formula = (softmax(Z) - Y) / len(labels)
numeric = np.zeros_like(Z)
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        E = np.zeros_like(Z)
        E[i, j] = 1e-5
        numeric[i, j] = (ce_of_Z(Z + E) - ce_of_Z(Z - E)) / 2e-5
print("(P − Y) / n coincide con el gradiente numérico:", np.allclose(formula, numeric, atol=1e-8))

## 7. Todo junto: un clasificador entrenado con gradiente

Clasificamos las tres especies con dos medidas: el largo del pico y el de la aleta, estandarizados. El modelo es una capa lineal (semana 2) seguida de *softmax*:

$$Z = X W + b, \qquad P = \text{softmax}(Z), \qquad X: (n, 2),\; W: (2, 3),\; b: (3,)$$

Por la regla de la cadena, con $G = (P - Y)/n$ el gradiente respecto de $Z$, los gradientes de los parámetros son $\nabla_W L = X^\top G$ y $\nabla_b L = \sum_i G_i$. Separamos los datos en entrenamiento y prueba (la semana 5 explica por qué es obligatorio) y comparamos con la línea base de la clase mayoritaria. Ejecuta la celda; la curva de pérdida aparece cuando completaste el Tu turno 5.

In [ ]:
species = penguins["species"].to_numpy()
classes = np.unique(species)                         # ['Adelie' 'Chinstrap' 'Gentoo']
labels = np.searchsorted(classes, species)           # 0, 1, 2
feats = penguins[["bill_length_mm", "flipper_length_mm"]].to_numpy()
feats = (feats - feats.mean(axis=0)) / feats.std(axis=0)

perm = np.random.default_rng(SEED).permutation(len(labels))  # partición fija, haya o no ejercicios resueltos
train, test = perm[:250], perm[250:]
X_tr, y_tr, X_te, y_te = feats[train], labels[train], feats[test], labels[test]
Y_tr = np.eye(3)[y_tr]

W = np.zeros((2, 3))
b = np.zeros(3)
lr = 1.0
history = []
for step in range(300):
    P = softmax(X_tr @ W + b)          # (250, 3)
    G = (P - Y_tr) / len(y_tr)         # ∂L/∂Z
    W -= lr * X_tr.T @ G               # (2, 250) @ (250, 3) → (2, 3)
    b -= lr * G.sum(axis=0)
    try:
        history.append(cross_entropy(P, y_tr))
    except NotImplementedError:
        pass

majority = np.bincount(y_tr).argmax()
acc_base = np.mean(y_te == majority)
acc = np.mean(softmax(X_te @ W + b).argmax(axis=1) == y_te)
print(f"línea base (siempre {classes[majority]}): accuracy de prueba {acc_base:.3f}")
print(f"softmax entrenada con gradiente:   accuracy de prueba {acc:.3f}")

fig, axes = plt.subplots(1, 2 if history else 1, figsize=(13 if history else 6.5, 4.8), squeeze=False)
ax = axes[0, 0]
gx, gy = np.meshgrid(np.linspace(-2.6, 2.8, 300), np.linspace(-2.3, 2.4, 300))
grid_pred = softmax(np.c_[gx.ravel(), gy.ravel()] @ W + b).argmax(axis=1).reshape(gx.shape)
ax.contourf(gx, gy, grid_pred, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#c6dbef", "#fdd0a2", "#c7e9c0"])
for c, color in zip(range(3), ["tab:blue", "tab:orange", "tab:green"]):
    ax.scatter(*X_te[y_te == c].T, s=18, color=color, label=classes[c])
ax.set_xlabel("largo del pico (estandarizado)")
ax.set_ylabel("largo de la aleta (estandarizado)")
ax.set_title("Regiones de decisión y pingüinos de prueba")
ax.legend()
if history:
    ax = axes[0, 1]
    ax.plot(history, color="tab:orange", lw=2)
    ax.axhline(np.log(3), color="gray", ls="--", label="ln 3: probabilidades uniformes")
    ax.set_xlabel("paso")
    ax.set_ylabel("entropía cruzada (entrenamiento)")
    ax.set_title("Curva de aprendizaje")
    ax.legend()
plt.tight_layout()
plt.show()

La pérdida empieza exactamente en $\ln 3$: con $W$ y $b$ en cero, todos los puntajes son iguales y la *softmax* reparte un tercio a cada clase. Después baja, y el clasificador deja muy atrás a la línea base. Todo con las piezas de hoy: un gradiente, la regla de la cadena y un bucle que da pasos cuesta abajo. Una red neuronal es esto mismo con más capas; PyTorch (semana 8) calcula los gradientes por nosotros.

## 8. Cierre

**Hoy vimos:**

- La derivada es una pendiente: cuánto cambia la salida por cada poquito que se mueve la entrada. La diferencia centrada la aproxima y sirve para verificar derivadas hechas a mano.
- Derivadas parciales: una variable a la vez, las demás quietas. El gradiente las junta y apunta cuesta arriba.
- Regla de la cadena: en una composición, las derivadas de los eslabones se multiplican (engranajes). Es la base de *backpropagation*.
- La pérdida es una función de los parámetros, con los datos fijos. Entrenar es buscar su mínimo.
- Descenso de gradiente: $\theta \leftarrow \theta - \eta \nabla L$. Con $\eta$ muy chica no llega; con $\eta$ demasiado grande diverge.
- *Softmax* convierte puntajes en probabilidades; la entropía cruzada es la sorpresa promedio, $-\ln$ de la probabilidad de la clase correcta. Línea base: $\ln k$.

**Para la semana 5:**

1. **Entrega al inicio de la clase**: **B2: Descenso de gradiente a mano** ([enunciado](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/B2_descenso_gradiente.ipynb)): descenso de gradiente en 1-D y 2-D, gráficos de contornos, y romperlo a propósito con una mala tasa de aprendizaje. Se guarda en tu repositorio de entregas como `B2/B2_descenso_gradiente.ipynb`.
2. **Lectura previa** (MÓLÓ, en inglés): [Supervised Learning](https://colab.research.google.com/drive/140FnxeOBXsbFYKNvJP8WwQ7XH45LNZ1W), [Inference, Training, Test and Evaluation](https://colab.research.google.com/drive/1H0AbbfDlyfmJDrIDcXVOqgC6OO87VO7g) y [Generalisation, Overfitting and Bias](https://colab.research.google.com/drive/1pzug6uPG9Ee_2dxRdzvymyzrpoUyfOGD).
3. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).

La semana 5 es ML clásico en síntesis: entrenamiento, validación y prueba, sobreajuste, métricas y sklearn. Al final se publica el Sprint 0.